In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from langchain_community.vectorstores import Chroma

In [21]:
import numpy
from typing import List

In [22]:
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

with open("sample_docs", "w", encoding="utf-8") as f:
    for content in sample_docs:
        f.write(content.strip() + "\n\n")


In [23]:
import os
os.environ["OPENAI_API_KEY"] ="Your key"

In [24]:
persist_directory = "./chroma_db"

In [25]:
class RagArch:
  def __init__(self,chunk_size = 1000, chunk_overlap = 10):
    self.chunk_size = chunk_size,
    self.chunk_overlap = chunk_overlap
    self.text_splitters = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
  def processor(self, file_path:str)-> List[Document]:
    lod = TextLoader(file_path)
    data = lod.load()

    processod_chunks = []

    for i, doc in enumerate(data):
      cleaned_data = " ".join(doc.page_content.split())
      chunks = self.text_splitters.create_documents([cleaned_data])
      processod_chunks.extend(chunks)

    vector_store = Chroma.from_documents(
        documents = processod_chunks,
        embedding = OpenAIEmbeddings(model = "text-embedding-3-small"),
        persist_directory = persist_directory,
        collection_name = "rag_collection"
      )
    return vector_store

In [26]:
rag = RagArch()

vectors = rag.processor("/content/sample_docs")

print(f"vectorstore created : {vectors._collection.count()}vectors")

vectorstore created : 6vectors


In [27]:
query = "what are the types of Machine Learning"

In [28]:
similar_docs = vectors.similarity_search(query, k=3)

similar_docs

[Document(metadata={}, page_content='Machine Learning Fundamentals Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties. Deep Learning and Neural Networks Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep learning has revolutionized fields like computer vision, natural language processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly effective for image processing, while Recurrent Neural Networks (R

In [22]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai: gpt-4o-mini")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7db0cfe35b50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7db0cfe1fd70>, root_client=<openai.OpenAI object at 0x7db0cfe1fc80>, root_async_client=<openai.AsyncOpenAI object at 0x7db0cfe1cf50>, model_name=' gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [29]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

llm.invoke("what is LLM?")


AIMessage(content='LLM stands for "Large Language Model." It refers to a type of artificial intelligence model that is designed to understand and generate human language. These models are trained on vast amounts of text data and use deep learning techniques, particularly neural networks, to learn patterns, grammar, facts, and even some reasoning abilities from the data.\n\nLLMs can perform a variety of language-related tasks, including:\n\n- Text generation\n- Translation\n- Summarization\n- Question answering\n- Sentiment analysis\n- Conversational agents (chatbots)\n\nExamples of large language models include OpenAI\'s GPT-3 and GPT-4, Google\'s BERT, and others. These models have gained significant attention for their ability to produce coherent and contextually relevant text, making them useful in various applications across industries.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 12, 'total_tokens': 170, 'comp

In [30]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [32]:
retriver = vectors.as_retriever(
    search_kwarg={"k":3}
)

retriver

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7f9c958fd010>, search_kwargs={})

In [34]:
system_prompt = """
You are an assitant for question and answering tasks.
Use the following retrived context to answer the question
If you don't know the answer say I don't know
keep answer in 3 sentences and precise

context : {context}"""

prompt = ChatPromptTemplate([
    ("system",system_prompt),
    ("human","{input}")]
)

In [35]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assitant for question and answering tasks.\nUse the following retrived context to answer the question\nIf you don't know the answer say I don't know\nkeep answer in 3 sentences and precise\n\ncontext : {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [36]:
document_chain = create_stuff_documents_chain(llm,prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assitant for question and answering tasks.\nUse the following retrived context to answer the question\nIf you don't know the answer say I don't know\nkeep answer in 3 sentences and precise\n\ncontext : {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7f9cb16fcef0>, async_client=<openai.resources.chat.completions.completions.AsyncComp

In [37]:
rag_chain = create_retrieval_chain(retriver, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7f9c958fd010>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assitant for question and answering tasks.\nUse the following retrived context to answer the question\nIf you don't know the an

In [39]:
response = rag_chain.invoke({"input":"What is deep learning"})

In [40]:
response["answer"]

'Deep learning is a subset of machine learning that is based on artificial neural networks, which are inspired by the human brain. These networks consist of layers of interconnected nodes and have significantly advanced fields such as computer vision, natural language processing, and speech recognition. Deep learning techniques, like Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs), are particularly effective for tasks involving image processing and sequential data.'